# Notebook 5B: Evaluating Classification Models with CalEnviroScreen

*Authored by Dr. Noelle Anderson in 2026*

## Introduction

In Notebook 5A, you changed the supervised learning question from regression to classification. Instead of predicting the numerical `Asthma` rate directly, you created a binary label called `High Asthma`.

In this notebook, we will keep using the same logistic regression model setup, but we will focus on a new question:

**How do we evaluate a classification model?**

This is different from evaluating the regression model in Notebook 4B. In regression, the model predicted a number, so we used metrics like RMSE and R². In classification, the model predicts a class label, so we need metrics that compare predicted labels with true labels.

Even though **logistic regression** has the word regression in its name, it is being used here as a **classification model**. It predicts probabilities first, then turns those probabilities into class labels.

By the end of this notebook, you should be able to:

- rebuild the same classification setup from Notebook 5A
- explain why classification needs different metrics than regression
- impute missing feature values after the train/test split
- scale features after imputation
- refit the logistic regression model as `log_model`
- calculate accuracy, precision, recall, and F1 score
- read a confusion matrix
- explain false positives and false negatives in a public-health context
- explain why threshold choices create tradeoffs
- separate model performance from causal explanation

## Notebook 5B roadmap

This notebook has three main parts.

**Part 1: Preprocess the data for classification**

1. Load the CalEnviroScreen dataset
2. Prepare rows with known `Asthma` values
3. Rebuild the `High Asthma` classification label
4. Choose the same features from Notebook 5A
5. Split the data
6. Impute and scale the features

**Part 2: Fit the logistic regression model**

7. Fit `log_model`
8. Make class predictions and probability predictions

**Part 3: Evaluate the classification model**

9. Calculate accuracy
10. Read a confusion matrix
11. Calculate precision, recall, and F1 score
12. Explore threshold tradeoffs
13. Put the metrics in context

## Why classification uses different metrics than regression

A **metric** is a number that summarizes one part of model performance.

In Notebook 4B, the linear regression model predicted a numerical asthma rate, so we used regression metrics:

- **RMSE** summarized the size of the prediction errors
- **R²** summarized how much variation in the target the model captured

In this notebook, the logistic regression model predicts a class label:

- `0` means lower asthma burden
- `1` means high asthma burden

Because the output is a class label, we need classification metrics. Classification metrics ask questions like:

- How often did the model predict the correct class?
- When the model predicted high asthma burden, how often was it right?
- Of the tracts that really had high asthma burden, how many did the model find?

These questions matter in biology, biochemistry, public health, and conservation because classification models are often used to sort cases into groups, like positive vs negative, high risk vs lower risk, or protected habitat vs lower-priority habitat.

## Step 0: Import the packages we need

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay # New today!
from sklearn.metrics import precision_score, recall_score, f1_score # New today!

# Part 1: Preprocess the data for classification

## Step 1: Load the CalEnviroScreen data

We will load the data from the same public Google Drive link you used in the previous notebooks. The code is provided so we can focus today's work on classification metrics.

In [ ]:
# Instructor-provided data loading code
file_id = "1X4-6X3VKhR2jRHppI3XuVV79nhHyg8Xb"
url = f"https://drive.google.com/uc?export=download&id={file_id}"

ces = pd.read_csv(url, index_col="Census Tract")

print("Dataset shape:", ces.shape)
display(ces.head())

## Step 2: Prepare rows with known `Asthma` values

In Notebooks 4A and 5A, you saw that we need to think carefully about column roles before modeling.

For this notebook, we will keep the dataframe intact and choose model inputs directly using `model_features`. That means columns can remain in the dataframe even if they are not used as inputs to today’s model.

The first cleaning step is about rows, not columns. We will remove rows where the original `Asthma` value is missing, because we cannot create the `High Asthma` label without the asthma value.


### <font color=0D9488>**Question 1:**</font> Create `ces_sup_clean` by dropping rows where `Asthma` is missing. Print the shape before and after dropping those rows.

In [ ]:
# Your solution here!

Next, we will rebuild the classification label from Notebook 5A.

We will use the median asthma rate as the cutoff. This is a teaching choice, not a medical or policy threshold. The median usually creates two classes that are close to balanced, which makes this first classification evaluation easier to learn.

### <font color=0D9488>**Question 2:**</font> Find the median `Asthma` value, save it as `asthma_cutoff`, and create the binary `High Asthma` label.

In [ ]:
# Your solution here!

### <font color=0D9488>**Question 3:**</font> Check the class counts and class proportions for `High Asthma`. Are the two classes close to balanced?

In [ ]:
# Your solution here!

Because we used the median as the cutoff, the two classes should be close to balanced. **Class balance** means checking how many rows are in each class.

Class balance matters because a model can look better than it really is if one class is much more common than the other. For example, if 95% of rows were class `0`, a model could get high accuracy by predicting class `0` almost all the time. That would not mean it was useful for finding class `1` rows.

## Step 3: Choose the same feature set from Notebook 5A

We will use the same broader non-race/ethnicity feature set from Notebook 5A. This lets us focus on evaluation without changing the modeling question again.

These features include environmental burden variables, community-condition variables, and age-structure variables. That does not mean the variables are simple causes of asthma burden. They are model inputs that may help with prediction, and many of them reflect broader structural conditions.

Some columns remain in `ces_sup_clean` but are not selected in `model_features` today. Context columns like county, ZIP code, and approximate location identify places more than they describe tract conditions for this first model. Columns like `Asthma Pctl`, `Pop. Char. Score`, and `Draft CES 5.0 Score` are also not selected because they are too closely tied to asthma or summarize multiple parts of the dataset.

Race/ethnicity percentage columns are important for environmental justice, but they are not part of today’s `model_features`. This keeps the model question focused while still recognizing that racialized patterns matter. Those patterns can reflect racism, segregation, environmental policy, housing policy, labor conditions, and unequal access to resources and protection. These structural conditions can contribute to asthma burden, but race itself is not a biological cause of asthma.

Later analyses could ask different questions about race/ethnicity columns, but that would require especially careful interpretation and framing.


In [ ]:
# Instructor-provided feature set for the classification model
model_features = [
    "Ozone",
    "PM2.5",
    "Diesel PM",
    "Drinking Water",
    "Pesticides",
    "Traffic",
    "Cleanup Sites",
    "Education",
    "Housing Burden",
    "Linguistic Isolation",
    "Poverty",
    "Unemployment",
    "Children < 10 years (%)",
    "Elderly > 64 years (%)"
]

## Step 4: Create `X` and `y`

Now we separate the data by column role.

- `X` contains only the selected columns in `model_features`
- `y` contains the classification target, `High Asthma`

Columns that are not listed in `model_features` remain in the dataframe, but they are not used as inputs to this model.

### <font color=0D9488>**Question 4:**</font> Create the `X` feature dataset using `model_features` and create the `y` target labels using `High Asthma`. Then print their shapes.

In [ ]:
# Your solution here!

## Step 5: Split the rows into training and test sets

We will use `train_test_split()` again.

Because this is classification, we will also use `stratify=y`. This tells sklearn to keep the class proportions similar in the training and test sets.

In [ ]:
# Instructor-provided classification train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
# Instructor-provided class proportion check
class_balance_check = pd.DataFrame({
    "Training proportion": y_train.value_counts(normalize=True).sort_index(),
    "Test proportion": y_test.value_counts(normalize=True).sort_index()
})

display(class_balance_check)

The proportions should be very similar because we used `stratify=y`. This helps make the test set a fairer check of how the model performs on both classes.

## Step 6: Impute missing feature values

The broader feature set has some missing values. That is common in real datasets.

In supervised learning, missing feature values should be handled **after** the train/test split. Otherwise, the test rows can influence the values we learn during preprocessing.

**Imputation** means filling in missing feature values using a chosen rule. Today, we will use median imputation, which fills missing values in each feature with the median value from that feature.

Median imputation is simple and useful for learning, but it has limitations. It does not recover the true missing values, and it can hide patterns in why values are missing.

The `SimpleImputer` tool we'll use can also use other basic strategies, like mean, most frequent value, or a constant value. More complex imputation methods also exist, but median imputation is enough for today's first classification-evaluation workflow.

In [ ]:
# Instructor-provided missingness check before imputation
print("Missing values in X_train before imputation:")
display(X_train.isna().sum())

print("Missing values in X_test before imputation:")
display(X_test.isna().sum())

We will not ask you to write the imputation code independently yet. This is the first time we are applying imputation in the modeling workflow, so the code is provided.

In [ ]:
# Instructor-provided imputation step

# Create an imputer that fills missing values with the median of each feature
imputer = SimpleImputer(strategy="median")

# Fit on the training data only, then transform the training data
X_train_imputed_array = imputer.fit_transform(X_train)

# Transform the test data using only the medians learned from X_train
X_test_imputed_array = imputer.transform(X_test)

# Convert the arrays back to dataframes so the column names stay visible
X_train_imputed = pd.DataFrame(
    X_train_imputed_array,
    columns=X_train.columns,
    index=X_train.index
)

X_test_imputed = pd.DataFrame(
    X_test_imputed_array,
    columns=X_test.columns,
    index=X_test.index
)

In [ ]:
# Instructor-provided check after imputation
print("Missing values in X_train after imputation:", X_train_imputed.isna().sum().sum())
print("Missing values in X_test after imputation:", X_test_imputed.isna().sum().sum())

Notice the pattern. It is similar to the supervised scaling pattern from Notebook 5A:

- create a preprocessing object
- fit it on the training data only
- transform the training data
- transform the test data using what was learned from the training data

For imputation, the learned values are the training-set medians. For scaling, the learned values are the training-set means and standard deviations.

## Step 7: Scale the features

After imputation, we will scale the features. You have now seen this supervised scaling pattern before, so this time you will write the full scaling step.

Remember:

- use `fit_transform()` on the training data
- use `transform()` only on the test data
- keep the scaled data in dataframes so the column names stay visible

### <font color=0D9488>**Question 5:**</font> Scale `X_train_imputed` and `X_test_imputed`. Save the scaled dataframes as `X_train_scaled` and `X_test_scaled`.

In [ ]:
# Your solution here!

# Part 2: Fit the logistic regression model

## Step 8: Fit `log_model`

Now we can fit the same kind of model from Notebook 5A.

Logistic regression learns coefficients because it is a linear model. Those coefficients are harder to interpret than linear regression coefficients because they relate to log-odds, not directly to the predicted probability. We will not interpret the coefficients today because the main goal is classification evaluation.

### <font color=0D9488>**Question 6:**</font> Create a logistic regression model called `log_model`, then use `.fit()` to train it on `X_train_scaled` and `y_train`.

In [ ]:
# Your solution here!

## Step 9: Make predictions

To evaluate the model, we need predictions on the test set.

We will create two kinds of predictions:

- predicted class labels with `predict()`
- predicted probabilities with `predict_proba()`

Classification metrics mostly use the predicted class labels. Threshold tradeoffs use the predicted probabilities.

### <font color=0D9488>**Question 7:**</font> Use `log_model.predict()` to make class predictions for `X_test_scaled`. Save them as `class_preds`.

In [ ]:
# Your solution here!

In [ ]:
# Instructor-provided probability predictions
predicted_probs = log_model.predict_proba(X_test_scaled)
high_asthma_probs = predicted_probs[:, 1]

prediction_df = pd.DataFrame({
    "Actual label": y_test,
    "Predicted label": class_preds,
    "Predicted probability of high asthma": high_asthma_probs
})

display(prediction_df.head(10))

# Part 3: Evaluate the classification model

## Step 10: Accuracy

**Accuracy** is the proportion of test rows that the model classified correctly. In plain language:

**accuracy = how often the model got the class label right overall**

Accuracy is useful, but it is incomplete by itself. It tells us how often the model was correct overall, but it does not tell us what kinds of mistakes the model made.

It also does not show whether the model is doing equally well for high-asthma-burden tracts and lower-asthma-burden tracts, but it's a great first pass metric.

Let's use sklearn's `accuracy_score()` to calculate the model's accuracy on the test set.

In [ ]:
accuracy = accuracy_score(y_test, class_preds)

print("Accuracy:", round(accuracy, 2))

A single accuracy score tells us how often the model was correct overall. It does not tell us whether the model is better at finding high-asthma-burden tracts or lower-asthma-burden tracts.

That is why we need the confusion matrix!

## Step 11: Confusion matrix

A **confusion matrix** may sound confusing, but it's just a table that sorts classification predictions into correct and incorrect categories.

For our target, remember:

- class `0` means lower asthma burden
- class `1` means high asthma burden

The confusion matrix helps us see four possible outcomes:

- **true negative:** actually class `0`, predicted class `0`
- **false positive:** actually class `0`, predicted class `1`
- **false negative:** actually class `1`, predicted class `0`
- **true positive:** actually class `1`, predicted class `1`

In [ ]:
# Instructor-provided confusion matrix plot
# Notice the inputs!
ConfusionMatrixDisplay.from_predictions(
    y_test,
    class_preds,
    cmap="Blues"
)
plt.title("Confusion Matrix for High Asthma Classification")
plt.show()

### <font color=0D9488>**Question 8:**</font> In this asthma-burden example, what would a false negative mean? What would a false positive mean? Which squares of the confusion matrix are those represented by?

**Your solution here!**

## Step 12: Precision, recall, and F1 score

Accuracy asks: **How often was the model correct overall?**

In terms of the confusion matrix:

**Accuracy = (TP + TN) / (TP + TN + FP + FN)**

Accuracy counts both kinds of correct predictions:

- **TP:** tracts that really were high asthma burden and were predicted as high asthma burden
- **TN:** tracts that really were lower asthma burden and were predicted as lower asthma burden

Accuracy is useful, but it does not tell us what kind of mistakes the model made. For that, we need metrics that focus more closely on class `1`, which here is the high-asthma-burden class.

One helpful way to remember precision and recall is:

- **Precision** starts from the model's positive predictions.
- **Recall** starts from the real positive cases.



---

**Precision** asks: When the model predicted high asthma burden, how often was it right?

**Precision = TP / (TP + FP)**

Precision compares true positives to all predicted positives. It is lower when the model creates more false positives.

A practical way to think about precision is: **How much should we trust a high-burden prediction?**

---

**Recall** asks: Of the tracts that truly had high asthma burden, how many did the model find?

**Recall = TP / (TP + FN)**

Recall compares true positives to all actual positives. It is lower when the model creates more false negatives.

A practical way to think about recall is: **How many of the high-burden tracts did we catch?**

These two metrics answer different questions. In public health, recall can be especially important when missing high-burden communities would be costly or harmful. Precision can be especially important when follow-up resources are very limited and false alarms are costly.

---

**F1 score** combines precision and recall into one score.

**F1 = 2 × (precision × recall) / (precision + recall)**

F1 is useful when we want one number that balances precision and recall. It can be helpful for comparing models, but it is less intuitive than looking at precision and recall separately.

I know this is quite a few new metrics at once, take your time understanding them and [see this video for more suppport](https://www.youtube.com/watch?v=8d3JbbSj-I8).

Let's use the sklearn tools `precision_score()`, `recall_score()`, and `f1_score()` to calculate precision, recall, and F1 scores:

In [ ]:
# Conveniently these tools are built in for us in sklearn!

precision = precision_score(y_test, class_preds)
recall = recall_score(y_test, class_preds)
f1 = f1_score(y_test, class_preds)

print("Precision:", precision)
print("Recall:", recall)
print("F1 score:", f1)

In [ ]:
# Metric summary table
metric_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 score"],
    "Value": [accuracy, precision, recall, f1]
})

# Round metric values to 3 decimal places for easier reading
metric_summary["Value"] = metric_summary["Value"].round(3)

display(metric_summary)

### What do we consider "good performance"?

Similar to in regression, there is no single cutoff that tells us whether a classification model is "good." A model's performance depends on the dataset, the question, and how the predictions might be used.

Here, the model has an accuracy of about 0.74, which means it correctly classified about 74% of the test rows. That is better than random guessing, but accuracy alone does not tell the whole story.

The precision is about 0.77, which means that when the model predicted high asthma burden, it was correct about 77% of the time. The recall is about 0.69, which means the model found about 69% of the tracts that really were in the high asthma burden group.

Whether that is "good enough" depends on the purpose. If the model were being used as an early screening tool to identify communities that may need more attention, we might care a lot about recall because missing high-burden tracts could be harmful. If follow-up resources were very limited, we might also care about precision because too many false positives could spread those resources too thin.

For this notebook, we should interpret this model as a reasonable first classification baseline. It is finding meaningful patterns, but it is not perfect, and its mistakes matter. In real public-health work, model performance should be evaluated alongside community knowledge, data quality, ethics, and the consequences of different kinds of errors.

### <font color=0D9488>**Question 9:**</font> If we are most worried about missing tracts that really are high asthma burden, should we pay more attention to precision or recall?

**Your solution here!**

## Step 13: Threshold tradeoffs

In Notebook 5A, you learned that logistic regression predicts probabilities first. Then it turns those probabilities into class labels using a threshold.

The default threshold is 0.5:

- probability below 0.5 becomes class `0`
- probability at or above 0.5 becomes class `1`

Changing the threshold changes the model's behavior.

A lower threshold usually predicts more rows as class `1`. This may catch more true high-asthma-burden tracts, but it may also create more false positives.

A higher threshold usually predicts fewer rows as class `1`. This may reduce false positives, but it may also miss more true high-asthma-burden tracts.

This kind of tradeoff appears across scientific fields. In public health screening, a lower threshold may help identify more people or communities that need follow-up. In biochemistry, a threshold might decide whether an assay result is treated as positive or negative. In conservation, a threshold might decide which habitats are flagged for protection. In each case, the threshold is not just technical. It affects which mistakes become more likely.

In [ ]:
# Instructor-provided threshold comparison
thresholds = [0.3, 0.5, 0.7]
threshold_results = []

for threshold in thresholds:
    threshold_preds = (high_asthma_probs >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Predicted class 1 count": threshold_preds.sum(),
        "Precision": precision_score(y_test, threshold_preds),
        "Recall": recall_score(y_test, threshold_preds)
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df)

### <font color=0D9488>**Question 10:**</font> Compare the 0.3, 0.5, and 0.7 thresholds. What happens to the number of predicted class `1` rows as the threshold increases? What happens to recall?

**Your solution here!**

We don't have time to cover the visualization tools that support choosing thresholds, but [see this page for more detail](https://developers.google.com/machine-learning/crash-course/classification/roc-and-auc)

## Step 14: Put the metrics in context

Metrics are tools for summarizing model behavior, they are not the same thing as truth, fairness, or causality.

This is especially important for environmental health data. A classification model may help us notice patterns, but it does not explain by itself why asthma burden is higher in some communities. Features like pollution, housing burden, poverty, education, unemployment, and linguistic isolation reflect larger systems, including racism, segregation, policy decisions, labor conditions, disinvestment, and unequal access to resources and protection.

A model can be useful for prediction while still being incomplete as an explanation.

# Congratulations, you have completed today's notebook!

## Key Takeaways:

- You rebuilt the same classification setup from Notebook 5A
- You reviewed why logistic regression is used for classification even though it has regression in its name
- You imputed missing feature values after the train/test split
- You scaled the imputed features using the supervised preprocessing pattern
- You refit the logistic regression model as `log_model`
- You made class predictions and probability predictions
- You calculated accuracy, precision, recall, and F1 score
- You read a confusion matrix
- You interpreted false positives and false negatives in a public-health context
- You explored how changing the threshold changes model behavior
- You practiced separating prediction, evaluation, and causal explanation

In this notebook, you learned that classification evaluation is not just about asking whether a model is right or wrong overall. Different metrics highlight different kinds of performance, and those differences matter when model predictions could shape biological, public-health, environmental, or conservation decisions.

## Where This Fits in the ML Workflow

This notebook completed the first full classification evaluation workflow:

1. start with rows that have a known target value
2. create a classification label
3. choose features for this model
4. split into training and test sets
5. impute missing feature values after splitting
6. scale features after imputation
7. fit a classification model
8. make predictions
9. evaluate the predictions with classification metrics
10. interpret the results carefully

In Week 6, we will move to tree-based models. These models can capture more flexible patterns than linear models, but they still require careful evaluation and careful interpretation.